In [35]:
from paths import *
from nano_maker.nanomaker import NanoMaker
from nano_maker.container.configs import (skeleton_weight_filename, naanobot_weight_filename,
                                          skeleton_config, naanobot_config, radial_config)

In [36]:
skeleton_weight_filename = skeleton_weight_filename
skeleton_cfg = skeleton_config
naanobot_weight_filename = naanobot_weight_filename
naanobot_config = naanobot_config
radial_cfg = radial_config

In [37]:
model = NanoMaker(skeleton_weight_filename=skeleton_weight_filename,
                  naanobot_weight_filename=naanobot_weight_filename,
                  skeleton_cfg=skeleton_config,
                  naanobot_config=naanobot_config,
                  radial_cfg=radial_cfg)

In [38]:
drug_i_want_to_deliver = "C[N+]1(C)C[C@@H]2C[C@H]1CN2c1ccc(nn1)-c1ccccc1 |r|"

In [39]:
model.ingest_chemical(drug_i_want_to_deliver)

Chemical Ingested: C[N+]1(C)C[C@@H]2C[C@H]1CN2c1ccc(nn1)-c1ccccc1 |r|
Drug Likeness Rules Passed: 4 / 4


In [58]:
pocket_data = model.generate_pocket_data(sampling_temp=0.01)
print(len(pocket_data))
print(type(pocket_data))

4
<class 'dict'>


In [59]:
pocket_data

{'SMILES': 'C[N+]1(C)C[C@@H]2C[C@H]1CN2c1ccc(nn1)-c1ccccc1 |r|',
 'Sampling_temperature': 0.01,
 '3D_skeleton': [[-13.559153789422886, 7.8646264186362105, 0.7212024905973644],
  [-14.119661508940071, 5.375276993641554, 1.01432726595441],
  [-13.70085213671254, 4.453815372409592, 3.366700483591679],
  [-13.789444404332015, 2.622521858961096, 3.174723452221727],
  [-14.052739593212753, 1.4596411139341297, 0.9818475900253363],
  [-13.882947915393071, 1.2846447115901174, 0.2974627760342272],
  [-13.482357839780763, 2.1240123545250507, 1.344436395738203],
  [-13.175206695894216, 2.3208561753810257, 1.5206464784634943],
  [-11.951667397488896, 5.7728349411583215, 1.8372818986382542],
  [-12.528622798490618, 4.099666294844277, 1.566965264351355],
  [-1.2746691711098923, 12.036597511894811, 5.494682333938168],
  [-12.542248498339372, 2.6929421528061406, 1.7562590287656268],
  [-0.6327077291398666, 11.600254487911013, 5.648789850074943],
  [-11.129040852507739, 5.515177627840127, 2.117172761691

In [60]:
smiles_str = pocket_data["SMILES"]
skeleton = pocket_data['3D_skeleton']
aa_seq = pocket_data['aa_ids']

In [61]:
# GENERATION QUALITY ASSESSMENTS

In [62]:
import torch
import torch.nn.functional as F
from nano_maker.naanobot import NAAnoBot
from nano_maker.modules.nAAno.smiles_handler import smiles_fingerprint

In [63]:
@torch.no_grad()
def diagnose_generation(model, map4_enc, sph_coordinates, n=120, sampling_temperature=0.2, verbose=True):
    model.eval()

    map4_enc = map4_enc.to(next(model.parameters()).device)

    bioch_context = [model.naano_module.get_nAAnovector("VOID") for _ in range(model.block_size)]
    coord_context = [[model.max_angstroms, 0, 0] for _ in range(model.block_size)]

    aa_chain = ""
    position_log = []

    # precompute normalized AA matrix once
    aa_ids = [aa_id for aa_id in model.naano_module.nAAno_vectors.keys() if aa_id != "VOID"]
    n_v_tensor = torch.tensor(
        [model.naano_module.nAAno_vectors[aa_id] for aa_id in aa_ids],
        dtype=torch.float32
    )
    aa_norm = F.normalize(n_v_tensor, dim=-1)  # [n_aa, features]

    for i, coordinate in enumerate(sph_coordinates[:n]):
        naano_X = model.naano_module.get_nAAno_X(
            coord_context, bioch_context, coordinate
        ).unsqueeze(0).to(map4_enc.device)

        output, _ = model.forward(naano_X, map4_enc)
        vector = output[0].detach().float()

        # cosine similarities — consistent with loss and approx_id
        pred_norm = F.normalize(vector.unsqueeze(0), dim=-1)
        similarities = (pred_norm @ aa_norm.T).squeeze(0)
        ranked = sorted(zip(aa_ids, similarities.tolist()), key=lambda x: -x[1])
        top5_names = [aa for aa, _ in ranked[:5]]
        greedy = top5_names[0]

        # sample
        sampled = model.naano_module.approx_id(output[0], sampling_temperature=sampling_temperature)
        diverged = sampled != greedy

        if verbose:
            top5_str = "  ".join(top5_names)
            diverge_str = f"  → sampled {sampled}" if diverged else ""
            print(f"[{i:2d}] r={coordinate[0]:.1f}Å  top5: {top5_str}{diverge_str}")

        bioch_context = bioch_context[1:] + [model.naano_module.get_nAAnovector(sampled)]
        coord_context = coord_context[1:] + [coordinate]
        aa_chain += sampled

        position_log.append({
            "position": i,
            "coordinate": coordinate,
            "greedy": greedy,
            "sampled": sampled,
            "top5": top5_names,
            "diverged": diverged
        })

    if verbose:
        divergences = sum(1 for p in position_log if p["diverged"])
        print(f"\nchain:      {aa_chain}")
        print(f"diverged:   {divergences}/{len(position_log)} ({100*divergences/len(position_log):.0f}%)")

    return aa_chain, position_log

In [64]:
map4_enc = torch.tensor(smiles_fingerprint(drug_i_want_to_deliver), dtype=torch.float32).unsqueeze(0)
nb_cfg = naanobot_config.copy()
sph_coordinates =[
    [18.42,  1.832,  0.847],
    [16.91, -2.441,  2.103],
    [15.73,  0.291,  1.234],
    [14.88, -1.057,  0.612],
    [13.55,  2.718,  2.441],
    [12.34, -0.883,  1.876],
    [11.92,  1.204,  0.394],
    [10.67, -2.093,  2.718],
    [ 9.44,  0.751,  1.047],
    [ 8.83, -1.571,  0.628],
    [ 7.62,  2.094,  2.356],
    [ 6.51, -0.314,  1.309],
    [ 5.38,  1.047,  0.785],
    [ 4.22, -2.618,  2.094],
    [ 3.14,  0.524,  1.571],
]

naanobot_model = NAAnoBot(n_embd=nb_cfg["n_embd"], n_head=nb_cfg["n_head"],
                                           n_layers=nb_cfg["n_layers"],
                                           block_size=nb_cfg["block_size"],
                                           map4_res=nb_cfg["map4_res"],
                                           n_spatial_features=nb_cfg["n_spatial_features"],
                                           max_angstroms=nb_cfg["max_angstroms"],
                                           dropout=nb_cfg["dropout"],
                          loss_temperature=nb_cfg["loss_temperature"])
diagnose_generation(model=model._NAAnoBotPrototype, map4_enc=map4_enc, sph_coordinates=sph_coordinates)

[ 0] r=18.4Å  top5: N  R  Q  H  D  → sampled K
[ 1] r=16.9Å  top5: R  K  Y  N  H  → sampled H
[ 2] r=15.7Å  top5: D  E  N  S  Q  → sampled E
[ 3] r=14.9Å  top5: Y  R  K  W  H  → sampled E
[ 4] r=13.6Å  top5: D  E  S  N  T  → sampled E
[ 5] r=12.3Å  top5: D  E  N  S  Q  → sampled H
[ 6] r=11.9Å  top5: T  A  V  L  S  → sampled D
[ 7] r=10.7Å  top5: D  S  N  T  E  → sampled N
[ 8] r=9.4Å  top5: A  G  T  H  V
[ 9] r=8.8Å  top5: R  K  Y  T  A  → sampled Y
[10] r=7.6Å  top5: T  S  H  N  A  → sampled E
[11] r=6.5Å  top5: S  G  T  A  N  → sampled T
[12] r=5.4Å  top5: S  G  T  N  A  → sampled I
[13] r=4.2Å  top5: A  G  T  V  S
[14] r=3.1Å  top5: S  N  T  G  Q  → sampled G

chain:      KHEEEHDNAYETIAG
diverged:   13/15 (87%)


('KHEEEHDNAYETIAG',
 [{'position': 0,
   'coordinate': [18.42, 1.832, 0.847],
   'greedy': 'N',
   'sampled': 'K',
   'top5': ['N', 'R', 'Q', 'H', 'D'],
   'diverged': True},
  {'position': 1,
   'coordinate': [16.91, -2.441, 2.103],
   'greedy': 'R',
   'sampled': 'H',
   'top5': ['R', 'K', 'Y', 'N', 'H'],
   'diverged': True},
  {'position': 2,
   'coordinate': [15.73, 0.291, 1.234],
   'greedy': 'D',
   'sampled': 'E',
   'top5': ['D', 'E', 'N', 'S', 'Q'],
   'diverged': True},
  {'position': 3,
   'coordinate': [14.88, -1.057, 0.612],
   'greedy': 'Y',
   'sampled': 'E',
   'top5': ['Y', 'R', 'K', 'W', 'H'],
   'diverged': True},
  {'position': 4,
   'coordinate': [13.55, 2.718, 2.441],
   'greedy': 'D',
   'sampled': 'E',
   'top5': ['D', 'E', 'S', 'N', 'T'],
   'diverged': True},
  {'position': 5,
   'coordinate': [12.34, -0.883, 1.876],
   'greedy': 'D',
   'sampled': 'H',
   'top5': ['D', 'E', 'N', 'S', 'Q'],
   'diverged': True},
  {'position': 6,
   'coordinate': [11.92, 1.20